# 01 · Data download and check

**Purpose:** download MIMIC-IV into the shared Drive folder (once), then load it through PyHealth's `InHospitalMortalityMIMIC4` task and print **aggregate**
statistics: samples, patients, deaths, hours per sample, features. Optionally make a smaller copy of
`labevents` with only the task's 27 lab codes, so later loads are faster.

> **Data rules:** never print, display or save raw MIMIC-IV rows. Only aggregate counts and scores.
> Don't ask Colab's AI assistant about cells that touch the data. Never commit data or data outputs to GitHub.

## Setup (run every session)

Change `DRIVE_FOLDER` to where the team's shared folder appears in **your** Drive.
A folder someone shared with you shows under "Shared with me": right-click it and choose
**Organize → Add shortcut** to My Drive so Colab can see it.

In [ ]:
# --- Settings: edit these ---
DRIVE_FOLDER = "/content/drive/MyDrive/DL4H_TPC"  # shared team folder
BRANCH = "tpc-dev"
REPO_URL = "https://github.com/avnimohan9/PyHealth.git"

import os, sys
assert sys.version_info >= (3, 12), f"PyHealth needs Python 3.12+, this runtime has {sys.version}"

from google.colab import drive
drive.mount("/content/drive")

MIMIC4_ROOT = f"{DRIVE_FOLDER}/mimiciv/2.2"      # contains hosp/ and icu/
RESULTS_DIR = f"{DRIVE_FOLDER}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ["MIMIC4_ROOT"] = MIMIC4_ROOT
os.environ["TPC_CACHE_DIR"] = "/content/tpc_cache"  # local disk, faster than Drive

In [ ]:
# Get the latest code from GitHub (clone the first time, pull after that)
if os.path.exists("/content/PyHealth"):
    !git -C /content/PyHealth pull -q
else:
    !git clone -q -b {BRANCH} {REPO_URL} /content/PyHealth
%cd /content/PyHealth
!git log --oneline -1
!pip install -q -e .
# If "import pyhealth" fails right after the first install: Runtime -> Restart session, then rerun setup.

## One-time: download MIMIC-IV into the shared Drive folder

**Only one person needs to run this, once.** It downloads the 5 files we need (about 3 GB, mostly
`labevents`) from PhysioNet straight into `MIMIC4_ROOT`. You type your own PhysioNet password when
asked; it is never saved or printed. Files already downloaded are skipped, so it's safe to rerun
after a disconnect.

In [ ]:
import getpass, subprocess

PHYSIONET_USER = "your-physionet-username"  # TODO
BASE = "https://physionet.org/files/mimiciv/2.2"
FILES = ["hosp/patients.csv.gz", "hosp/admissions.csv.gz", "hosp/d_labitems.csv.gz",
         "icu/icustays.csv.gz", "hosp/labevents.csv.gz"]

password = getpass.getpass("PhysioNet password: ")
for f in FILES:
    out = f"{MIMIC4_ROOT}/{f}"
    if os.path.exists(out):
        print("already there:", f)
        continue
    os.makedirs(os.path.dirname(out), exist_ok=True)
    tmp = out + ".part"
    r = subprocess.run(["wget", "-q", "-c", "--user", PHYSIONET_USER, "--password", password,
                        "-O", tmp, f"{BASE}/{f}"])
    if r.returncode == 0:
        os.replace(tmp, out)
        print("downloaded:", f, f"({os.path.getsize(out) / 1e9:.2f} GB)")
    else:
        print("FAILED:", f, "- check your username/password and that you signed the MIMIC-IV data use agreement")
del password

## Load the samples
Uses `load_samples()` from the example script once it's implemented.

In [ ]:
sys.path.insert(0, "/content/PyHealth/examples")
from mimic4_mortality_tpc import load_samples

samples = load_samples()
# TODO: print aggregates only, e.g.
#   number of samples, unique patients, deaths and death rate
#   min / median / max hours per sample
#   number of lab features

## Optional: filtered `labevents` copy

Full `labevents` is very large. A copy with only the task's 27 lab codes is much smaller and faster to
load. Save it in the shared Drive folder (credentialed team only) and point `MIMIC4_ROOT` at it.

In [ ]:
from pyhealth.tasks import InHospitalMortalityMIMIC4
LAB_IDS = [int(i) for i in InHospitalMortalityMIMIC4.LABITEMS]
FILTERED_ROOT = f"{DRIVE_FOLDER}/mimiciv_filtered/2.2"
# TODO: read hosp/labevents in chunks (pandas, chunksize=...), keep rows whose itemid is in LAB_IDS,
# write FILTERED_ROOT/hosp/labevents.csv.gz, and copy the other four files unchanged.